# DATA CLEANING

In [111]:
import pandas as pd

# Define your tables
table_names = [
    'orders',
    'customers', 
    'products',
    'sellers',
    'order_payments',
    'order_items',
    'order_reviews',
    'geoloc'
]

# Load all into a dictionary
datasets = {
    name: pd.read_parquet(f'../data/outputs/{name}_raw.parquet')
    for name in table_names
}

### Validity - Assigning relevant data types

In [112]:
# Orders 

datasets['orders']['order_status'] = datasets['orders']['order_status'].astype('category')
datasets['orders']['order_purchase_timestamp'] = pd.to_datetime(datasets['orders']['order_purchase_timestamp'])
datasets['orders']['order_approved_at'] = pd.to_datetime(datasets['orders']['order_approved_at'])
datasets['orders']['order_delivered_carrier_date'] = pd.to_datetime(datasets['orders']['order_delivered_carrier_date'])
datasets['orders']['order_delivered_customer_date'] = pd.to_datetime(datasets['orders']['order_delivered_customer_date'])
datasets['orders']['order_estimated_delivery_date'] = pd.to_datetime(datasets['orders']['order_estimated_delivery_date'])

# Customers 

datasets['customers']['customer_city'] = datasets['customers']['customer_city'].astype('category')
datasets['customers']['customer_state'] = datasets['customers']['customer_state'].astype('category')

# Products - Everything is fine for this dataset

# Sellers
datasets['sellers']['seller_city'] = datasets['sellers']['seller_city'].astype('category')
datasets['sellers']['seller_state'] = datasets['sellers']['seller_state'].astype('category')

# Order payments
datasets['order_payments']['payment_type'] = datasets['order_payments']['payment_type'].astype('category')

# Order items
datasets['order_items']['order_item_id'] = datasets['order_items']['order_item_id'].astype('str')
datasets['order_items']['shipping_limit_date'] = pd.to_datetime(datasets['order_items']['shipping_limit_date'])

# Order reviews
datasets['order_reviews']['review_creation_date'] = pd.to_datetime(datasets['order_reviews']['review_creation_date'])
datasets['order_reviews']['review_answer_timestamp'] = pd.to_datetime(datasets['order_reviews']['review_answer_timestamp'])

# Geo-location
datasets['geoloc']['geolocation_city'] = datasets['geoloc']['geolocation_city'].astype('category')
datasets['geoloc']['geolocation_state'] = datasets['geoloc']['geolocation_state'].astype('category')

### Completeness - dealing with missing value

In [113]:
datasets['orders'].isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [116]:
## Orders
# order_approved_at - will be imputed with median values only for the orders that were delivered
orders = datasets['orders']
median_timestamp = (orders['order_approved_at'] - orders['order_purchase_timestamp'] ).median()
orders['order_approved_at'] = orders[orders['order_status'] == 'delivered']['order_approved_at'].fillna(median_timestamp)